In [ ]:
import pickle

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import numpy as np
import pandas as pd

# Data Preparation

In [ ]:
df = pd.read_excel("data.xlsx", header=0)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df = df.fillna("")

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
def clean_text(text):

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    text = text.lower()

    return text

In [ ]:
def clean_title_rus(row):
    return clean_text(row["title_rus"])

In [ ]:
df["payload"] = df.apply(clean_title_rus, axis=1)

# Data Modelling

## BM25

In [ ]:
import bm25s

In [ ]:
corpus = df["payload"].tolist()

In [ ]:
corpus_tokens = bm25s.tokenize(corpus)

In [ ]:
retriever = bm25s.BM25(
    corpus=corpus,
    method="robertson",
    k1=1.5,
    b=0.75,
)

In [ ]:
retriever.index(corpus_tokens)

In [ ]:
def bm25s_search(query: str, top_k: int = 5):
    query_tokens = bm25s.tokenize(query)
    docs_idx, scores = retriever.retrieve(query_tokens, k=top_k)

    docs_idx = docs_idx[0]
    scores = scores[0]

    return pd.DataFrame(
        data=zip(docs_idx, scores, ["bm25s"] * top_k),
        columns=["title", "score", "algo"],
    )

In [ ]:
for row in bm25s_search("движение капитала", top_k=5).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

In [ ]:
with open("output/bm25s_retriever.pkl", "wb") as f:
    pickle.dump(retriever, f)

##  TF‑IDF + Cosine

In [ ]:
from stop_words import get_stop_words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
ru_stopwords = get_stop_words("russian")
eng_stopwords = get_stop_words("english")

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=1,
    stop_words=list(set(ru_stopwords + eng_stopwords)),
)

In [ ]:
tfidf = vectorizer.fit_transform(df["payload"])

In [ ]:
def search_by_query(query: str, top_k: int = 5):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return pd.DataFrame(
        data=zip(
            df.iloc[top_idx]["payload"].to_list(), sims[top_idx], ["tfidf"] * top_k
        ),
        columns=["title", "score", "algo"],
    )

In [ ]:
for row in search_by_query("движение капитала", top_k=5).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

In [ ]:
with open("output/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
with open("output/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

## SentenceTransformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
corpus_texts = df["payload"].tolist()

In [ ]:
item_embeddings = model.encode(corpus_texts, normalize_embeddings=True)

In [ ]:
def semantic_search(query: str, top_k: int = 5):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, item_embeddings).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return pd.DataFrame(
        data=zip(
            df.iloc[top_idx]["payload"].to_list(),
            sims[top_idx],
            ["transformer"] * top_k,
        ),
        columns=["title", "score", "algo"],
    )

In [ ]:
for row in semantic_search("движение капитала", top_k=3).iterrows():
    if row[1]["score"] > 0:
        print(f"{row[1]["algo"]} - {round(row[1]["score"], 2)} - {row[1]["title"]}")
        print()

In [ ]:
with open("output/item_embeddings.pkl", "wb") as f:
    pickle.dump(item_embeddings, f)

In [ ]:
with open("output/model.pkl", "wb") as f:
    pickle.dump(model, f)

## Ranker

In [ ]:
query = "движение капитала"

In [ ]:
dfs = []

In [ ]:
for foo in [bm25s_search, search_by_query, semantic_search]:
    dfs.append(foo(query, 3))

In [ ]:
output = pd.concat(dfs).reset_index(drop=True)

In [ ]:
output = output.pivot_table(
    index="title",
    columns="algo",
    values="score",
    fill_value=0,
)

In [ ]:
output.columns = ["bm25s", "tfidf", "transformer"]

In [ ]:
output.reset_index(inplace=True)

In [ ]:
output

In [ ]:
def simple_rank(candidates_df, w_bm25=0.2, w_tfidf=0.2, w_transformer=0.6):
    df = candidates_df.copy()

    for col in ["bm25s", "tfidf", "transformer"]:
        m, M = df[col].min(), df[col].max()
        if M > m:
            df[col + "_norm"] = (df[col] - m) / (M - m)
        else:
            df[col + "_norm"] = 0.0

    df["final_score"] = (
        w_bm25 * df["bm25s_norm"]
        + w_tfidf * df["tfidf_norm"]
        + w_transformer * df["transformer_norm"]
    )

    df = df.sort_values("final_score", ascending=False).reset_index(drop=True)

    return df[["title", "bm25s_norm", "tfidf_norm", "transformer_norm", "final_score"]]

In [ ]:
simple_rank(output)